# 04 · Human–agent collaboration and a critic agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smusleh/LLM_Bioinformatics_Agents/blob/main/notebooks/04_human_in_loop_critic.ipynb)

**Paper concepts:**
- **Human–agent collaboration** (Fig. 3, third mode): the human approves or redirects the agents' plan at checkpoints, as in MEDAGENTS and BioMaster, where people and agents work in layers.
- **Verification / critic roles**: a critic agent, like the one in the ProtAgents framework, checks the report against the raw tool outputs. This addresses the hallucination and interpretability challenges.

Flow: **report from notebook 03 → PI proposes follow-up → 🧑 human gate → team executes → critic audits → 🧑 human gate → revised final report**

> Set `AUTO_APPROVE=1` in the environment to run this notebook headless. Otherwise `input()` prompts you, and that also works inside VS Code notebooks.

In [ ]:
# ▶ Colab setup: run this cell first (it does nothing when you run the notebook locally).
# BACKEND: "ollama"    free, no account; use Runtime -> Change runtime type -> T4 GPU (first run ~3-5 min)
#          "gemini"    free key from aistudio.google.com/apikey, saved as Colab secret GEMINI_API_KEY
#          "anthropic" Claude; Colab secret ANTHROPIC_API_KEY + API credits
BACKEND = "ollama"
import os, sys, subprocess
if "google.colab" in sys.modules:
    REPO_DIR = "/content/LLM_Bioinformatics_Agents"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/smusleh/LLM_Bioinformatics_Agents.git", REPO_DIR], check=True)
    os.chdir(f"{REPO_DIR}/notebooks")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "anthropic==2.8.0", "openai>=1.50.0", "rdata==0.11.2", "python-dotenv"], check=True)
    sys.path.insert(0, REPO_DIR)
    from omics_agents import colab
    colab.setup(BACKEND)

In [ ]:
import sys, os, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make `omics_agents` importable
%load_ext autoreload
%autoreload 2
from IPython.display import Markdown, Image, display
import pandas as pd
from omics_agents import config, data, tools
from omics_agents import rag
from omics_agents.agent import Agent, human_gate
from omics_agents.team import make_team, make_pi

In [ ]:
if config.PROVIDER == "ollama":
    print("Backend: ollama | model:", config.OLLAMA_MODEL, "| url:", config.OLLAMA_BASE_URL)
else:
    assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in the .env file at the repo root (see .env.example)."
    print("Backend: anthropic | model:", config.MODEL, "| effort:", config.EFFORT)

In [ ]:
report_path = config.OUTPUT_DIR / "pi_report.md"
assert report_path.exists(), "Run notebook 03 first."
report = report_path.read_text(encoding="utf-8")
tool_log = json.loads((config.OUTPUT_DIR / "pi_tool_log.json").read_text(encoding="utf-8"))
print(f"Loaded report ({len(report)} chars) and {len(tool_log)} tool calls")

## 1. PI proposes a follow-up plan → human approval

In [ ]:
team = make_team()
pi = make_pi(team)
plan = pi.run(
    "Here is our previous report:\n\n" + report + "\n\nPropose (do NOT execute yet) a short "
    "numbered plan of at most 3 follow-up analyses that would most strengthen these conclusions. "
    "Do not call any tools for this step.")
display(Markdown(plan))

In [ ]:
decision = human_gate("Approve the PI's follow-up plan?")
instruction = ("The human approved your plan. Execute it now." if decision.lower() in ("y", "yes")
               else f"The human redirected the plan. Follow these instructions instead: {decision}")
followup = pi.run(instruction + " Then write an updated full report (markdown).")
display(Markdown(followup))

## 2. A critic agent audits the report against the raw tool outputs

In [ ]:
all_logs = tool_log + [{"agent": k, **e} for k, a in team.items() for e in a.tool_log]
evidence = json.dumps([{k: e[k] for k in ("agent", "tool", "input", "result")} for e in all_logs])
CRITIC = (
    "You are a critical reviewer of computational biology reports. You receive a report and the "
    "complete JSON log of tool calls that produced it. Identify: (a) numbers not present in the "
    "log or misquoted, (b) biological claims without KB support (check with search_knowledge), "
    "(c) overreaching conclusions (e.g. causal language from correlations, n=150 caveats). "
    "Return a numbered list of issues with severity (major/minor) and a suggested fix. "
    "Be specific; do not invent problems."
)
critic = Agent("Critic", CRITIC, toolbox=tools.Toolbox(
    [], extra={"search_knowledge": (rag.search_knowledge, rag.SEARCH_SCHEMA)}))
critique = critic.run(f"REPORT:\n{followup}\n\nTOOL LOG (JSON):\n{evidence}")
display(Markdown(critique))

## 3. Human decides which critiques to act on → final revision

In [ ]:
decision = human_gate("Ask the PI to revise the report addressing ALL critic issues?")
guidance = ("Address all issues." if decision.lower() in ("y", "yes")
            else f"Human guidance on which issues to address: {decision}")
final = pi.run(f"A critic reviewed your report:\n\n{critique}\n\n{guidance} Do not call tools "
               "unless needed to fix a number. Output only the final revised markdown report.")
(config.OUTPUT_DIR / "report.md").write_text(final, encoding="utf-8")
display(Markdown(final))

In [ ]:
agents = {"PI": pi, "Critic": critic, **team}
print(f"Total ~ ${sum(a.cost() for a in agents.values()):.3f}")

## Connecting back to the paper's *Challenges and future directions*

| Challenge in the paper | Where it showed up in this demo | Mitigation demonstrated |
|---|---|---|
| Model hallucination & interpretability | LLM-only baseline (01); biology without evidence (02) | Tool-grounded numbers, RAG citations, verifier and critic agents, full tool logs |
| Updating and timeliness of knowledge | Fixed, hand-curated knowledge base | Retrieval separates knowledge from model weights; long-term memory (02) |
| Limitations of agent architecture | Overhead from PI delegation (03) | Narrow specialist toolboxes, clear interfaces between modules |
| Omics heterogeneity (sparsity, noise, naming) | Protein missing in test set; layers with different noise | Late fusion with uncertainty weighting; layer-aware specialists |
| Data privacy & security | Summaries are sent to a cloud LLM | Agents receive only aggregate statistics, never patient-level rows |
| Ethical / legal responsibility | Who signs off on conclusions? | Human gates at plan and final-report stages |